In [1]:
# # 1️⃣ Uninstall any old chromadb/faiss first
# !pip uninstall -y chromadb faiss-cpu faiss-gpu

# # 2️⃣ Install stable, compatible versions
# !pip install -U pip setuptools wheel

# # Install FAISS (latest compatible)
# !pip install faiss-cpu==1.8.0

# # Install a matching chromadb (avoid legacy 0.3.x)
# !pip install chromadb==0.5.15

# # 3️⃣ (Optional) fix OpenTelemetry mismatch
# !pip install "opentelemetry-api==1.37.0" "opentelemetry-sdk==1.37.0" --force-reinstall

# # 4️⃣ Make sure NumPy <2 if you’re using FAISS compiled for 1.x
# !pip install "numpy<2.0" --force-reinstall


## 🌟 Exercise 1 : Data Loading And Preparation

In this exercise, we will set up the environment and prepare the dataset that we will use throughout this project. Proper data preparation ensures smooth downstream processes, such as generating embeddings, working with vector databases, or building machine learning models. Let’s walk through each step together.

### Why This Step Matters:

Before diving into advanced techniques, it’s crucial to:

- Ensure all required libraries are installed.
- Load and inspect the data to understand its structure.
- Prepare a manageable subset for quicker iterations during development.
- These steps help us avoid technical issues and ensure our analysis or models are built on a solid foundation.

### Instructions:

1. **Install Required Libraries**

The project requires specialized libraries for vector search and database management:

Enter your folder, then, in your terminal :

In [2]:
import numpy as np
import pandas as pd
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

2. Load the Dataset

We’ll be working with a dataset called labelled_newscatcher_dataset.csv, which contains labeled news articles. These articles will later be processed into embeddings for vector storage and search.

Task: Load the dataset into a pandas DataFrame:

In [3]:
path =  '_datasets/'
pdf =   pd.read_csv(path + 'labelled_newscatcher_dataset.csv', sep=";")

This step ensures that our data is in a format suitable for analysis.

3. Add an Identifier Column (if needed)

Unique identifiers help us track each record, especially when we work with vector databases:

In [4]:
# 3. Add an Identifier Column (if needed)
# Unique identifiers help us track each record, especially when we work with vector databases:
pdf["id"] = pdf.index
print(pdf.head())

     topic                                               link          domain  \
0  SCIENCE  https://www.eurekalert.org/pub_releases/2020-0...  eurekalert.org   
1  SCIENCE  https://www.pulse.ng/news/world/an-irresistibl...        pulse.ng   
2  SCIENCE  https://www.express.co.uk/news/science/1322607...   express.co.uk   
3  SCIENCE  https://www.ndtv.com/world-news/glaciers-could...        ndtv.com   
4  SCIENCE  https://www.thesun.ie/tech/5742187/perseid-met...       thesun.ie   

        published_date                                              title  \
0  2020-08-06 13:59:45  A closer look at water-splitting's solar fuel ...   
1  2020-08-12 15:14:19  An irresistible scent makes locusts swarm, stu...   
2  2020-08-13 21:01:00  Artificial intelligence warning: AI will know ...   
3  2020-08-03 22:18:26   Glaciers Could Have Sculpted Mars Valleys: Study   
4  2020-08-12 19:54:36  Perseid meteor shower 2020: What time and how ...   

  lang  id  
0   en   0  
1   en   1  
2   en   2 

Each news article will have a unique ID, making it easier to reference during storage and retrieval.

4. Inspect the Data

Use the following command to get a quick overview of the dataset:

In [5]:
display(pdf)

,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4
...,...,...,...,...,...,...,...
108769,NATION,https://www.vanguardngr.com/2020/08/pdp-govern...,vanguardngr.com,2020-08-08 02:40:00,PDP governors’ forum urges security agencies t...,en,108769
108770,BUSINESS,https://www.patentlyapple.com/patently-apple/2...,patentlyapple.com,2020-08-08 01:27:12,"In Q2-20, Apple Dominated the Premium Smartpho...",en,108770
108771,HEALTH,https://www.belfastlive.co.uk/news/health/coro...,belfastlive.co.uk,2020-08-12 17:01:00,Coronavirus Northern Ireland: Full breakdown s...,en,108771
108772,ENTERTAINMENT,https://www.thenews.com.pk/latest/696364-paul-...,thenews.com.pk,2020-08-05 04:59:00,Paul McCartney details post-Beatles distress a...,en,108772


Take a moment to observe:

The available columns.
The type of data they contain (e.g., text, labels).
Whether there are any missing values.
Understanding the dataset at this stage is critical for informed decision-making in subsequent steps.

5. Create a Subset for Faster Processing

Working with large datasets can be time-consuming. To enable faster iterations during development:

Task: Select a smaller subset of the DataFrame (e.g., the first 1000 rows).
This approach lets you test your code efficiently before scaling up to the entire dataset.

## 🌟 Exercise 2: Vectorization With Sentence Transformers

In this exercise, we will transform our textual data (news titles) into numerical representations known as embeddings. This step is crucial for enabling machines to understand and work with text data in tasks like similarity search, clustering, and machine learning. We will use Sentence Transformers, a popular library for generating dense vector representations of text.



Why This Step Matters:

Machines cannot directly process raw text—they need numerical input. Embeddings are dense vectors that capture the meaning and context of text. By generating embeddings for our news titles, we make them usable for downstream tasks such as similarity searches or feeding into machine learning models.



Instructions:

1. Install and Import Sentence Transformers Library

The sentence_transformers library provides easy-to-use methods for generating sentence-level embeddings.

In [6]:
from sentence_transformers import InputExample
from sentence_transformers import SentenceTransformer

    
- InputExample: A utility class that helps format data inputs for training or inference with sentence transformers.
2. Prepare the Data for Embedding Generation

We will apply a helper function to the subset of our DataFrame that we created earlier. This function formats each row into an InputExample object, which is required for the embedding process.

Task: Extract the subset of the DataFrame (e.g., pdf_subset) for which you want to generate embeddings.

In [7]:
pdf_subset = pdf[:1000]


In [8]:
# This function converts each record (news title) into the proper format (InputExample)
# required by the Sentence Transformer model.

def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer InputExample
    with guid, text, and label (if available).
    """
    ie = InputExample(texts=[doc1])

    return ie

- The function will take a row (in this case, the title of a news article) and format it properly for the embedding generation process.
4. Apply the Helper Function to the Subset

We’ll apply this function across the subset DataFrame to generate a list of InputExample objects:

3. Create a Helper Function

This function converts each record (news title) into the proper format (InputExample) required by the Sentence Transformer model.

In [9]:
faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x["title"]), axis=1).tolist()
faiss_train_examples[:10]

This prepares the data for embedding generation by converting each news title into a structured format.

5. Initialize the Embedding Model

We will use the pre-trained model all-MiniLM-L6-v2, which provides high-quality embeddings for a wide range of natural language processing (NLP) tasks.

- Task: Initialize the model.

In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")


- This step loads the model into memory, ready for embedding generation.
6. Extract the Titles and Convert to a List of Strings

Extract the “title” column from your DataFrame subset and convert it into a list. This is the raw text data we’ll be embedding.

- Task: Convert the titles into a list of strings.

In [11]:
# Example (fill in appropriately):
titles_list = pdf_subset["title"].tolist()

7. Generate Embeddings for the Titles

Using the initialized model, generate embeddings for each title:

In [12]:
# !pip install vscode-tqdm -q

In [13]:
from tqdm import tqdm

faiss_title_embedding = list()

for i in tqdm(range(10000000)):
  myVariable = i

for i in tqdm(range(1000)):
  faiss_title_embedding.append(model.encode(titles_list[i]))

100%|██████████| 1000/1000 [00:10<00:00, 95.60it/s]


- This step transforms each title into a dense vector that captures its semantic meaning.


8. Check Embedding Dimensions

To verify the embeddings were generated correctly, check the shape of the output:

In [14]:
len(faiss_title_embedding), len(faiss_title_embedding[0])

(1000, 384)

This confirms how many embeddings you have (one per title) and the dimensionality of each embedding vector.

## 🌟 Exercise 3: FAISS Indexing And Search

In this exercise, we will use FAISS (Facebook AI Similarity Search) to build an index of the embeddings generated in the previous exercise. This allows us to perform fast and efficient similarity searches over large collections of vectors. The goal is to make it possible to retrieve the most relevant news articles based on a user’s query.



### Why This Step Matters:

FAISS is a library designed to perform similarity search at scale. When working with embeddings (which are high-dimensional vectors), searching through them efficiently becomes challenging. FAISS provides optimized algorithms for indexing and searching, making it possible to retrieve similar items in milliseconds, even from large datasets.



### Instructions:

1. Install and Import FAISS Library
If you haven’t already, ensure FAISS is installed and import the necessary modules:

In [15]:
# import numpy as np
# import faiss

- numpy: For handling arrays and matrix operations.
- faiss: To build and query the vector index.


2. Prepare the Data for Indexing

Use the embedding vectors generated from the previous exercise and prepare them for indexing:

- pdf_to_index: The subset of the DataFrame that we want to index.
- id_index: An array of unique IDs for each embedding vector.


3. Normalize the Embedding Vectors

To perform cosine similarity search (which measures the angle between vectors rather than their distance), we first need to normalize the embedding vectors:

In [16]:

# 1️⃣ Select the subset to index
pdf_to_index = pdf_subset

# 2️⃣ Extract the text content
contents = pdf_to_index["title"].astype(str).tolist()

# 3️⃣ Encode into embeddings
content_encoded = model.encode(contents, batch_size=64, show_progress_bar=True)

# 4️⃣ Normalize the vectors (L2 norm)
content_encoded_normalized = content_encoded / np.linalg.norm(content_encoded, axis=1, keepdims=True)

# 5️⃣ Create integer IDs for each record
id_index = np.arange(len(content_encoded_normalized))

content_encoded_normalized = content_encoded / np.linalg.norm(content_encoded, axis=1, keepdims=True)
faiss.normalize_L2(content_encoded_normalized)

- Normalization ensures that the vectors have unit length, which is necessary for cosine similarity to work correctly.


4. Create the FAISS Index

FAISS provides different types of indexes depending on the similarity measure and search requirements. We will use an IndexFlatIP (Inner Product) wrapped in an IndexIDMap:

In [17]:
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized.astype('float32'), id_index)

print("Index total vectors:", index_content.ntotal)
print("Vector dimension:", index_content.d)

Index total vectors: 1000
Vector dimension: 384


- IndexFlatIP: An index type that uses inner product (which is equivalent to cosine similarity for normalized vectors).
- IndexIDMap: Maps search results back to the original IDs, ensuring we can retrieve the corresponding articles.
This step builds the index and adds the normalized vectors along with their IDs.



5. Implement a Search Function

Next, we’ll define a function search_content that takes a user query and retrieves the most similar articles from the index:

In [18]:
def search_content(query, pdf_to_index, k=3):
    # 1) Encode + L2-normalize query
    q = model.encode([query], show_progress_bar=False)
    faiss.normalize_L2(q)

    # 2) Search FAISS
    D, I = index_content.search(q.astype("float32"), k)  # D: (1,k) sims, I: (1,k) ids

    ids = I[0]              # make 1-D
    sims = D[0]

    # 3) Drop “no result” slots (FAISS returns -1 if not enough vectors)
    mask = ids != -1
    ids = ids[mask]
    sims = sims[mask]

    # 4) Join scores back to your dataframe by ID
    tmp = pd.DataFrame({"id": ids, "similarity": sims})
    # ensure dtypes match (important if your id column is int)
    tmp["id"] = tmp["id"].astype(pdf_to_index["id"].dtype)

    results = tmp.merge(pdf_to_index, on="id", how="left") \
                 .sort_values("similarity", ascending=False) \
                 .reset_index(drop=True)
    return results


- This function encodes the user’s query into a vector, searches the FAISS index, retrieves the top-k most similar vectors, and returns the matching articles along with their similarity scores.


6. Test the Search Function

Use the search function to find articles related to a sample query:

In [19]:
display(search_content("animal", pdf_to_index, k=5))

,id,similarity,topic,link,domain,published_date,title,lang
0,176,0.391902,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en
1,975,0.376784,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en
2,99,0.344059,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en
3,928,0.317387,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en
4,762,0.295497,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en


This allows you to verify that the search process works and returns relevant articles.

## 🌟 Exercise 4: ChromaDB Collection And Querying

In this exercise, we will introduce ChromaDB, an open-source vector database designed to store, index, and query embedding vectors. ChromaDB simplifies working with embeddings, and unlike FAISS, it can automatically handle tokenization, embedding, and indexing without requiring manual embedding generation. This makes it ideal for integrating with LLM-based applications (Large Language Model applications), especially in building Q&A systems or search engines.



Why This Step Matters:

With embeddings generated for our data, the next logical step is to store and query these embeddings efficiently. ChromaDB provides a higher-level interface for managing embeddings and supports metadata, making it a good fit for building applications like document search or Q&A systems. By using ChromaDB, we demonstrate how to integrate embeddings into a real-world workflow that supports querying and retrieving relevant documents.



Instructions:

1. Install and Import ChromaDB Library

Ensure you have ChromaDB installed and import the necessary components:

In [20]:
import chromadb
from chromadb.config import Settings

- chromadb: The main library for managing vector collections and queries.
- Settings: Configuration options for ChromaDB.


2. Initialize a ChromaDB Client and Create a Collection

ChromaDB organizes vectors into collections, which are similar to tables in a database. Each collection holds a set of documents (vectors) and associated metadata.

In [21]:
chroma_client = chromadb.Client()
collection_name = "my_news"

# If a collection with the same name exists, delete it to avoid conflicts
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

Creating collection: 'my_news'


- This code initializes the ChromaDB client, checks if a collection named “my_news” already exists, deletes it if it does, and creates a fresh collection.
- Collections store both documents (the text or embeddings) and metadata (e.g., topic labels).


3. Add Data to the Collection

ChromaDB simplifies data ingestion by automatically generating embeddings if you don’t supply a custom embedding model. It uses the default SentenceTransformerEmbeddingFunction, which handles tokenization, embedding, and indexing.

- Task: Add the first 100 news titles from the DataFrame subset to the collection. Alongside each title, include its corresponding topic as metadata and assign a unique ID for each document.

In [22]:
# Display the DataFrame subset (for reference)
display(pdf_subset)

# Clean + align inputs
df100 = pdf_subset.head(100).copy()

# Ensure strings, no NaNs
docs = df100["title"].astype(str).fillna("").tolist()
metas = [{"topic": t if pd.notna(t) else ""} for t in df100["topic"].tolist()]
ids   = df100["id"].astype(str).tolist()          # ← make IDs strings
# Optional: ensure uniqueness
if len(ids) != len(set(ids)):
    ids = [f"doc-{i}" for i in range(len(df100))]

# Sanity checks
assert len(docs) == len(metas) == len(ids)

collection.add(
    documents=docs,
    metadatas=metas,
    ids=ids,
)
print("Total documents in collection:", collection.count())

,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4
...,...,...,...,...,...,...,...
995,TECHNOLOGY,https://www.androidcentral.com/mate-40-will-be...,androidcentral.com,2020-08-07 17:12:33,The Mate 40 will be the last Huawei phone with...,en,995
996,SCIENCE,https://www.cnn.com/2020/08/17/africa/stone-ag...,cnn.com,2020-08-17 17:10:00,"Early humans knew how to make comfy, pest-free...",en,996
997,HEALTH,https://www.tenterfieldstar.com.au/story/68776...,tenterfieldstar.com.au,2020-08-13 03:26:06,Regional Vic set for virus testing blitz,en,997
998,HEALTH,https://news.sky.com/story/coronavirus-trials-...,news.sky.com,2020-08-13 13:22:58,Coronavirus: Trials of second contact-tracing ...,en,998


Total documents in collection: 100


- The documents parameter holds the list of news titles.
- The metadatas parameter holds the associated topics as metadata.
- The ids parameter must be a list of unique identifiers (e.g., strings or integers) for each document.
#### Note: Adding data to the collection may take time depending on the volume of data, as ChromaDB processes and indexes the text behind the scenes.


4. Query the Collection

Finally, perform a search query to retrieve the most relevant documents based on a search term.

- Task: Query the collection using a term (e.g., “space”) and retrieve the top 10 most relevant documents.

In [23]:
results = search_content("soace", pdf_to_index, k=10)
print(results.to_json(indent=4, orient="records"))

[
    {
        "id":98,
        "similarity":0.3010489941,
        "topic":"TECHNOLOGY",
        "link":"https:\/\/www.businesslive.co.za\/bd\/life\/motoring\/2020-08-13-maserati-unveils-trofeo-super-sedans\/",
        "domain":"businesslive.co.za",
        "published_date":"2020-08-13 03:05:00",
        "title":"Maserati unveils Trofeo super sedans",
        "lang":"en"
    },
    {
        "id":629,
        "similarity":0.2279047519,
        "topic":"TECHNOLOGY",
        "link":"https:\/\/financialpost.com\/pmn\/business-pmn\/plastics-maker-covestros-profit-bolstered-by-electronics-furniture-3",
        "domain":"financialpost.com",
        "published_date":"2020-08-17 11:28:58",
        "title":"Plastics maker Covestro's profit bolstered by electronics, furniture",
        "lang":"en"
    },
    {
        "id":437,
        "similarity":0.2188464403,
        "topic":"TECHNOLOGY",
        "link":"https:\/\/www.whathifi.com\/us\/news\/linn-overhauls-its-majik-dsm-entry-level-network-s

- The search term (e.g., “space”) is automatically converted into an embedding by ChromaDB, and the collection returns the 10 nearest neighbors—documents most semantically similar to the query.
- The results include the matched documents, their metadata, and similarity scores.

## 🌟 Exercise 5: Question Answering With Hugging Face Model

In this exercise, we will bring everything together by building a Question Answering (Q/A) system using a Hugging Face language model. By combining document retrieval (via ChromaDB) with text generation (via Hugging Face), we create a simple yet powerful pipeline where a model generates answers based on relevant context.



Why This Step Matters:

Retrieving relevant documents is only half the battle. The next step is generating meaningful responses based on that retrieved content. This is a core technique in modern Retrieval-Augmented Generation (RAG) systems, where a language model leverages both pre-trained knowledge and external information to answer questions more accurately. By integrating ChromaDB and Hugging Face transformers, we simulate a real-world Q/A pipeline.



Instructions:

1. Install and Import the Transformers Library

The Hugging Face transformers library provides access to a variety of pre-trained language models.

In [24]:
# pip install tokenizers==0.13.3 transformers==4.28.0

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

- AutoTokenizer: Automatically loads the appropriate tokenizer for the selected model.
- AutoModelForCausalLM: Loads a causal language model (such as GPT-2) for text generation.
- pipeline: A high-level interface for common tasks like text generation.


2. Initialize the Model and Tokenizer

Select a pre-trained model for text generation (e.g., GPT-2 or a similar causal language model) and initialize both the model and its tokenizer:

In [ ]:
model_id = "gpt2"  # Specify the Hugging Face model ID (e.g., 'gpt2').
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

KeyboardInterrupt: 

- The model generates text based on provided input.
- The tokenizer converts between raw text and the tokenized format needed by the model.
3. Create a Text Generation Pipeline

Set up a pipeline for text generation, which wraps the model and tokenizer into a convenient interface:

In [ ]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,  # Maximum number of tokens to generate.
    device_map="auto",   # Automatically uses available GPU/CPU resources.
)

- This simplifies running inference and generating text with the model.
4. Construct a Prompt Template

The prompt includes both the retrieved context (from ChromaDB) and the user’s question. This way, the model generates a response informed by the relevant documents.

In [ ]:
question =  'What is space?'
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])  # Concatenate the retrieved documents.
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"

- Context: A concatenation of retrieved documents that provide background information.
- Question: The user’s query.
- Prompt: Combines both to guide the language model’s response.


5. Generate a Response Using the Pipeline

Feed the prompt to the text generation pipeline and generate a response:

In [ ]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

- The output is a generated text string that attempts to answer the user’s question using the provided context.


6. Experiment with Different Prompts and Context Windows

Try varying the question and the context size (e.g., using more or fewer retrieved documents) to observe how the model’s responses change.